# Reproduce the detection ablation

This notebook reproduces, end to end, the two tables of the detection ablation (tile level on the test set and
mosaic level on the Diciembre orthomosaics) from the published checkpoints:

1. installs the exact environment used to produce the results,
2. downloads the two Kaggle datasets (tiles, checkpoints and orthomosaics),
3. evaluates the 16 detectors (8 architectures x 2 training sets) with their fixed, validation-selected
   score thresholds (`ablations/thresholds.json`),
4. builds the LaTeX tables and checks that every value matches the published ones.

**Runtime:** Colab with a GPU (*Runtime → Change runtime type → T4 GPU*). About 30–40 min, most of it installing
the environment and downloading the data. The Teselas/s column of the tables is the throughput measured on a
Tesla T4 (`ablations/benchmark_t4.json`); the last (optional) cell measures it on the current GPU.

## 1. Code and environment

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Mapping-rosettes" if IN_COLAB else os.getcwd()
if IN_COLAB and not os.path.exists(REPO):
    !git clone --branch reproduce https://github.com/pauzca/Mapping-rosettes.git {REPO}
%cd {REPO}

In [ ]:
# exact package versions used for the results (takes a few minutes)
if IN_COLAB:
    !pip install -q -r requirements.txt

## 2. Data from Kaggle

The datasets are private: add your Kaggle API token as a Colab secret named `KAGGLE_API_TOKEN`
(key icon on the left) or upload your `kaggle.json` when asked.

In [ ]:
DATA = os.environ.get("ROSETTES_DATA", "/content/data")
DATASETS = {"mixed": "mixed-resolution-frailejones", "lowres": "low-resolution-frailejones-detection"}

if not all(os.path.exists(f"{DATA}/{slug}/dataset/data.yaml") for slug in DATASETS.values()):
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
        except Exception:
            from google.colab import files
            print("Upload kaggle.json"); files.upload()
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    for slug in DATASETS.values():
        !kaggle datasets download paulauzcategui/{slug} -p {DATA}/{slug} --unzip -q
!ls {DATA}/*

## 3. Evaluate the 16 detectors

Each model is evaluated on the test tiles (per source) and on the two Diciembre orthomosaics, at its fixed
validation-selected threshold. Results go to `outputs/eval/<regime>/<model>/`.

In [ ]:
import json
THRESHOLDS = json.load(open("ablations/thresholds.json"))
ULTRALYTICS = {"yolo11s", "yolov8s", "yolo11m", "rtdetr_l"}
MOSAICS = f"{DATA}/{DATASETS['mixed']}/mosaics"

for regime, models in THRESHOLDS.items():
    root = f"{DATA}/{DATASETS[regime]}"
    for model, thr in models.items():
        ckpt = f"{root}/checkpoints/{model}"
        spec = f"ultralytics:{ckpt}/weights/best.pt" if model in ULTRALYTICS else f"run:{ckpt}"
        print(f"=== {regime} / {model} (threshold {thr})", flush=True)
        !python -m ablations.evaluate_model --model {spec} --thr {thr} --data {root}/dataset/data.yaml \
            --mosaics-root {MOSAICS} --out outputs/eval/{regime}/{model} 2>&1 | grep -E "^(test|dic|threshold)|Error"

## 4. Tables and check against the published values

In [ ]:
!python -m ablations.tables --eval-root outputs/eval
!python -m ablations.check --eval-root outputs/eval

In [ ]:
from IPython.display import Markdown, display
for name in ("tabla_teselas", "tabla_mosaicos"):
    display(Markdown(f"**{name}.tex**\n```latex\n{open(f'outputs/eval/{name}.tex').read()}\n```"))

## 5. (Optional) Throughput on this GPU

The tables use the throughput measured on a Tesla T4. This measures parameters and latency on the current
GPU for reference; it is not used to build the tables.

In [ ]:
specs = []
for model in THRESHOLDS["mixed"]:
    ckpt = f"{DATA}/{DATASETS['mixed']}/checkpoints/{model}"
    specs.append(f"{model}=" + (f"ultralytics:{ckpt}/weights/best.pt" if model in ULTRALYTICS else f"run:{ckpt}"))
!python -m ablations.benchmark --out outputs/eval/benchmark_this_gpu.json {" ".join(specs)} 2>&1 | grep -E "params_M|Error"